# Part D: Gender Classification

Project directions covered:
1. Gender classification from face images using CNNs
2. Modified CNN with depthwise-separable convolutions, dilated convolutions, and residual blocks
3. Comparison against a plain CNN baseline

**Dataset:** CelebA (`Male` attribute as gender label)  
**Models:** BaselineCNN vs ModifiedGenderCNN  
**Key technique:** Depthwise-separable conv + dilation=2 + residual shortcuts

In [ ]:
# Cell 1 — Install dependencies
!pip install -q torch torchvision pandas matplotlib scikit-learn seaborn gdown

In [ ]:
# Cell 2 — Imports
import os
import json
import random
from dataclasses import asdict, dataclass
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import seaborn as sns

from torch.utils.data import Dataset, DataLoader, Subset
from torchvision.datasets import CelebA
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report

In [ ]:
# Cell 3 — Config and reproducibility
SEED = 42
IMG_SIZE = 128
BATCH_SIZE = 64
EPOCHS = 5
LR = 1e-3
WEIGHT_DECAY = 1e-4
PATIENCE = 3

TRAIN_PER_CLASS = 2500   # balanced: 2500 female + 2500 male = 5000 train images
VAL_PER_CLASS   = 500
TEST_PER_CLASS  = 500

CLASS_NAMES = ["female", "male"]   # label 0 = female, 1 = male
OUTPUT_DIR  = "part_d_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

In [ ]:
# Cell 4 — Data augmentation and normalisation transforms
# Training: flip + rotation + colour jitter for regularisation
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.10),
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]),
])

# Val/test: only resize + normalise (no stochastic augmentation)
eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]),
])

# Download CelebA (may take a few minutes the first time)
print("Loading CelebA...")
train_base = CelebA(root="./data", split="train", target_type="attr", download=True, transform=train_transform)
val_base   = CelebA(root="./data", split="valid", target_type="attr", download=True, transform=eval_transform)
test_base  = CelebA(root="./data", split="test",  target_type="attr", download=True, transform=eval_transform)

male_idx = train_base.attr_names.index("Male")
print(f"Male attribute index: {male_idx}")
print(f"Full train set size : {len(train_base):,}")

In [ ]:
# Cell 5 — Dataset wrapper and balanced sampler
class GenderDataset(Dataset):
    """Wraps CelebA, returning (image, gender_label) where 0=female, 1=male."""
    def __init__(self, base_dataset, attr_index: int):
        self.base_dataset = base_dataset
        self.attr_index   = attr_index

    def __len__(self):
        return len(self.base_dataset)

    def __getitem__(self, idx):
        image, attrs = self.base_dataset[idx]
        label = int(attrs[self.attr_index] > 0)
        return image, torch.tensor(label, dtype=torch.long)


def balanced_subset(base_dataset, attr_index: int, samples_per_class: int):
    """Return indices for a class-balanced subset of size 2 * samples_per_class."""
    labels   = (base_dataset.attr[:, attr_index] > 0).long()
    selected = []
    for label in [0, 1]:
        indices   = torch.where(labels == label)[0]
        generator = torch.Generator().manual_seed(SEED + label)
        shuffled  = indices[torch.randperm(len(indices), generator=generator)]
        selected.extend(shuffled[:samples_per_class].tolist())
    return selected


# Build gender-labelled datasets and balanced subsets
train_dataset = GenderDataset(train_base, male_idx)
val_dataset   = GenderDataset(val_base,   male_idx)
test_dataset  = GenderDataset(test_base,  male_idx)

train_dataset = Subset(train_dataset, balanced_subset(train_base, male_idx, TRAIN_PER_CLASS))
val_dataset   = Subset(val_dataset,   balanced_subset(val_base,   male_idx, VAL_PER_CLASS))
test_dataset  = Subset(test_dataset,  balanced_subset(test_base,  male_idx, TEST_PER_CLASS))

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=2, pin_memory=torch.cuda.is_available())
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=2, pin_memory=torch.cuda.is_available())
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=2, pin_memory=torch.cuda.is_available())

print(f"Train: {len(train_dataset):,}  Val: {len(val_dataset):,}  Test: {len(test_dataset):,}")

In [ ]:
# Cell 6 — Visualise a batch of training samples
images, labels = next(iter(train_loader))

plt.figure(figsize=(12, 4))
for i in range(8):
    img = images[i].permute(1, 2, 0).numpy()
    img = np.clip((img * 0.5) + 0.5, 0, 1)   # undo normalisation
    plt.subplot(2, 4, i + 1)
    plt.imshow(img)
    plt.title(CLASS_NAMES[labels[i].item()])
    plt.axis("off")
plt.suptitle("Sample Training Images (female=0, male=1)")
plt.tight_layout()
plt.show()

In [ ]:
# Cell 7 — Model A: BaselineCNN
# Plain Conv -> ReLU -> MaxPool stack. No residuals, no dilation, no depthwise.
# Serves as the performance and parameter-count baseline for comparison.

class BaselineCNN(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),                          # 128 -> 64

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),                          # 64  -> 32

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),                          # 32  -> 16

            nn.AdaptiveAvgPool2d((1, 1)),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128, 64),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(64, num_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))


baseline_params = sum(p.numel() for p in BaselineCNN().parameters())
print(f"BaselineCNN trainable parameters: {baseline_params:,}")

In [ ]:
# Cell 8 — Model B: ModifiedGenderCNN
# Key improvements over baseline:
#   Depthwise-separable convolutions  -> fewer parameters, similar receptive field
#   Dilation=2 in second block        -> larger receptive field (captures face structure)
#   Residual shortcuts                -> easier gradient flow

class DepthwiseSeparableConv(nn.Module):
    """Factorises Conv(in, out, 3x3) into depthwise + pointwise.
    Reduces parameters ~8-9x at 3x3 while preserving expressiveness."""
    def __init__(self, in_channels, out_channels, dilation=1):
        super().__init__()
        padding = dilation   # same-padding for 3x3 kernel with dilation d
        self.block = nn.Sequential(
            # Depthwise: each channel filtered independently
            nn.Conv2d(in_channels, in_channels, kernel_size=3, padding=padding,
                      dilation=dilation, groups=in_channels, bias=False),
            nn.BatchNorm2d(in_channels),
            nn.ReLU(inplace=True),
            # Pointwise: mix channels with 1x1 conv
            nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)


class ResidualDSBlock(nn.Module):
    """Depthwise-separable conv block with a residual shortcut.
    1x1 projection aligns channel dims when they differ."""
    def __init__(self, in_channels, out_channels, dilation=1):
        super().__init__()
        self.conv = DepthwiseSeparableConv(in_channels, out_channels, dilation=dilation)
        if in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=False),
                nn.BatchNorm2d(out_channels),
            )
        else:
            self.shortcut = nn.Identity()
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        return self.relu(self.conv(x) + self.shortcut(x))


class ModifiedGenderCNN(nn.Module):
    """Gender classifier with depthwise-separable + dilated convolutions.
    Architecture:
      stem   : standard 3x3 conv (low-level edge detection)
      block1 : ResidualDSBlock(32->64,  dilation=1)
      block2 : ResidualDSBlock(64->128, dilation=2) — enlarged receptive field
      block3 : ResidualDSBlock(128->256,dilation=1)
      head   : 256 -> 128 -> 2 with dropout
    """
    def __init__(self, num_classes=2):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
        )
        self.features = nn.Sequential(
            nn.MaxPool2d(2),                              # 128 -> 64
            ResidualDSBlock(32,  64,  dilation=1),
            nn.MaxPool2d(2),                              # 64  -> 32
            ResidualDSBlock(64,  128, dilation=2),        # dilation=2: wider context
            nn.MaxPool2d(2),                              # 32  -> 16
            ResidualDSBlock(128, 256, dilation=1),
            nn.AdaptiveAvgPool2d((1, 1)),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.4),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(self.stem(x)))


modified_params = sum(p.numel() for p in ModifiedGenderCNN().parameters())
print(f"ModifiedGenderCNN trainable parameters : {modified_params:,}")
print(f"BaselineCNN       trainable parameters : {baseline_params:,}")
print(f"Parameter reduction: {(1 - modified_params/baseline_params)*100:.1f}%")

In [ ]:
# Cell 9 — Training history dataclass
@dataclass
class TrainHistoryRow:
    """One epoch of training metrics for one model."""
    model_name: str
    epoch:      int
    train_loss: float
    train_acc:  float
    val_loss:   float
    val_acc:    float
    lr:         float

print("TrainHistoryRow dataclass defined.")

In [ ]:
# Cell 10 — Training and evaluation loop functions
def train_one_epoch(model, loader, criterion, optimizer):
    """One full pass over the training set. Returns (avg_loss, accuracy)."""
    model.train()
    total_loss, all_labels, all_preds = 0.0, [], []

    for images, labels in loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        outputs = model(images)
        loss    = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        all_labels.extend(labels.detach().cpu().numpy())
        all_preds.extend(torch.argmax(outputs, dim=1).detach().cpu().numpy())

    return total_loss / len(loader), accuracy_score(all_labels, all_preds)


def evaluate(model, loader, criterion):
    """Evaluate model on a DataLoader. Returns (avg_loss, accuracy, true_labels, pred_labels)."""
    model.eval()
    total_loss, all_labels, all_preds = 0.0, [], []

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            outputs = model(images)
            loss    = criterion(outputs, labels)
            total_loss += loss.item()
            all_labels.extend(labels.detach().cpu().numpy())
            all_preds.extend(torch.argmax(outputs, dim=1).detach().cpu().numpy())

    return total_loss / len(loader), accuracy_score(all_labels, all_preds), all_labels, all_preds


def train_model(model_name: str, model: nn.Module):
    """Full training loop with early stopping and LR scheduling.
    Returns (checkpoint_path, history_df)."""
    print("=" * 80)
    print(f"Training {model_name}")

    model      = model.to(DEVICE)
    criterion  = nn.CrossEntropyLoss()
    optimizer  = optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler  = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=1)

    best_val_acc   = 0.0
    patience_ctr   = 0
    history        = []
    ckpt_path      = os.path.join(OUTPUT_DIR, f"best_{model_name}.pth")

    for epoch in range(1, EPOCHS + 1):
        train_loss, train_acc   = train_one_epoch(model, train_loader, criterion, optimizer)
        val_loss, val_acc, _, _ = evaluate(model, val_loader, criterion)
        scheduler.step(val_acc)

        row = TrainHistoryRow(
            model_name=model_name, epoch=epoch,
            train_loss=train_loss, train_acc=train_acc,
            val_loss=val_loss,     val_acc=val_acc,
            lr=optimizer.param_groups[0]["lr"],
        )
        history.append(asdict(row))

        print(f"Epoch {epoch}/{EPOCHS} | "
              f"train_loss={train_loss:.4f} train_acc={train_acc:.4f} | "
              f"val_loss={val_loss:.4f} val_acc={val_acc:.4f} | "
              f"lr={optimizer.param_groups[0]['lr']:.6f}")

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), ckpt_path)
            patience_ctr = 0
        else:
            patience_ctr += 1

        if patience_ctr >= PATIENCE:
            print("Early stopping triggered.")
            break

    history_df = pd.DataFrame(history)
    history_df.to_csv(os.path.join(OUTPUT_DIR, f"{model_name}_history.csv"), index=False)
    return ckpt_path, history_df


def test_model(model_name: str, model: nn.Module, ckpt_path: str):
    """Load best checkpoint and evaluate on the held-out test set."""
    criterion = nn.CrossEntropyLoss()
    model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE))
    model = model.to(DEVICE)

    test_loss, test_acc, y_true, y_pred = evaluate(model, test_loader, criterion)
    cm     = confusion_matrix(y_true, y_pred)
    report = classification_report(y_true, y_pred, target_names=CLASS_NAMES, digits=4)

    print(f"\n{model_name} | Test Loss: {test_loss:.4f}  Test Accuracy: {test_acc:.4f}")
    print(report)

    return {
        "model_name":            model_name,
        "test_loss":             float(test_loss),
        "test_accuracy":         float(test_acc),
        "test_f1":               float(f1_score(y_true, y_pred, average="macro")),
        "y_true":                y_true,
        "y_pred":                y_pred,
        "confusion_matrix":      cm.tolist(),
        "classification_report": report,
    }


print("Training utilities defined.")

In [ ]:
# Cell 11 — Train both models
# Warning: long-running cell — approx 8-15 min on Colab T4 GPU
baseline_ckpt, baseline_history = train_model("baseline_cnn",         BaselineCNN())
modified_ckpt, modified_history = train_model("modified_gender_cnn",  ModifiedGenderCNN())

In [ ]:
# Cell 12 — Training curves for both models on the same axes
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for hist, label, ls in [
    (baseline_history, "Baseline",  "--"),
    (modified_history, "Modified",  "-"),
]:
    axes[0].plot(hist["epoch"], hist["train_loss"], linestyle=ls, label=f"{label} train")
    axes[0].plot(hist["epoch"], hist["val_loss"],   linestyle=ls, label=f"{label} val", alpha=0.7)
    axes[1].plot(hist["epoch"], hist["train_acc"],  linestyle=ls, label=f"{label} train")
    axes[1].plot(hist["epoch"], hist["val_acc"],    linestyle=ls, label=f"{label} val",  alpha=0.7)

axes[0].set_title("Training & Validation Loss")
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Loss")
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].set_title("Training & Validation Accuracy")
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Accuracy")
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.suptitle("Baseline CNN vs Modified Gender CNN — Training Curves", fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "training_curves.png"), dpi=200)
plt.show()
print("Saved: training_curves.png")

In [ ]:
# Cell 13 — Test set evaluation for both models
baseline_results = test_model("baseline_cnn",        BaselineCNN(),        baseline_ckpt)
modified_results = test_model("modified_gender_cnn", ModifiedGenderCNN(),  modified_ckpt)

In [ ]:
# Cell 14 — Architecture comparison summary table
comparison_df = pd.DataFrame([
    {
        "model":         baseline_results["model_name"],
        "params":        baseline_params,
        "test_loss":     baseline_results["test_loss"],
        "test_accuracy": baseline_results["test_accuracy"],
        "test_f1":       baseline_results["test_f1"],
    },
    {
        "model":         modified_results["model_name"],
        "params":        modified_params,
        "test_loss":     modified_results["test_loss"],
        "test_accuracy": modified_results["test_accuracy"],
        "test_f1":       modified_results["test_f1"],
    },
]).sort_values("test_accuracy", ascending=False)

comparison_df.to_csv(os.path.join(OUTPUT_DIR, "model_comparison.csv"), index=False)
print("Model comparison:")
comparison_df

In [ ]:
# Cell 15 — Save metrics and classification reports to disk
for result in [baseline_results, modified_results]:
    name = result["model_name"]
    with open(os.path.join(OUTPUT_DIR, f"{name}_results.json"), "w") as f:
        json.dump({
            "model_name":       result["model_name"],
            "test_loss":        result["test_loss"],
            "test_accuracy":    result["test_accuracy"],
            "test_f1":          result["test_f1"],
            "confusion_matrix": result["confusion_matrix"],
        }, f, indent=2)
    with open(os.path.join(OUTPUT_DIR, f"{name}_report.txt"), "w") as f:
        f.write(result["classification_report"])
    print(f"Saved: {name}_results.json  +  {name}_report.txt")

In [ ]:
# Cell 16 — Bar chart: test accuracy comparison
plt.figure(figsize=(6, 4))
plt.bar(comparison_df["model"], comparison_df["test_accuracy"], color=["steelblue", "coral"])
plt.ylim(0, 1)
plt.title("Test Accuracy: Baseline vs Modified CNN")
plt.xlabel("Model")
plt.ylabel("Test Accuracy")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "test_accuracy_comparison.png"), dpi=200)
plt.show()
print("Saved: test_accuracy_comparison.png")

In [ ]:
# Cell 17 — Side-by-side confusion matrices
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

for ax, result in zip(axes, [baseline_results, modified_results]):
    cm = np.array(result["confusion_matrix"])
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax)
    ax.set_title(f"{result['model_name']}\n(Acc={result['test_accuracy']:.3f})")
    ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")

plt.suptitle("Gender Classification — Confusion Matrices (Test Set)", fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "confusion_matrices.png"), dpi=200)
plt.show()
print("Saved: confusion_matrices.png")

In [ ]:
# Cell 18 — Sample predictions from the best model (green=correct, red=wrong)
best_name  = comparison_df.iloc[0]["model"]
best_model = ModifiedGenderCNN() if best_name == "modified_gender_cnn" else BaselineCNN()
best_ckpt  = modified_ckpt       if best_name == "modified_gender_cnn" else baseline_ckpt

best_model.load_state_dict(torch.load(best_ckpt, map_location=DEVICE))
best_model = best_model.to(DEVICE)
best_model.eval()

images, labels = next(iter(test_loader))
with torch.no_grad():
    preds = torch.argmax(best_model(images.to(DEVICE)), dim=1).cpu()

plt.figure(figsize=(12, 4))
for i in range(8):
    img   = images[i].permute(1, 2, 0).numpy()
    img   = np.clip((img * 0.5) + 0.5, 0, 1)
    color = "green" if preds[i].item() == labels[i].item() else "red"
    plt.subplot(2, 4, i + 1)
    plt.imshow(img)
    plt.title(f"Pred: {CLASS_NAMES[preds[i].item()]}\nTrue: {CLASS_NAMES[labels[i].item()]}",
               color=color, fontsize=8)
    plt.axis("off")

plt.suptitle(f"Test Predictions — {best_name} (green=correct, red=wrong)", fontsize=12)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "sample_predictions.png"), dpi=200)
plt.show()
print("Saved: sample_predictions.png")
print("\nAll outputs saved in:", OUTPUT_DIR)